In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample, shuffle
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import random

# Define content path
content_path = "../data/"

/home/azureuser/NLP_CW/nlp_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# making results determinsistic
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)  # Hugging Face helper

## Data Loading and Preprocessing

In [3]:
# Load the dataset and preprocess it for training.

print("--- Loading and Preprocessing Data ---")
# Full data
df_full = pd.read_csv(
    content_path+"dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=4,
    header=None
)
df_full.columns = ["par_id", "art_id", "keyword", "country_code", "text", "label"]

# Train data
df_train_raw = pd.read_csv(content_path+"train_semeval_parids-labels.csv")
train_df  = df_full.merge(
    df_train_raw.drop(columns=['label']),
    on='par_id', how='inner'
)
train_df = train_df[['text', 'label']].dropna(subset=['text'])
train_df['label'] = train_df['label'].apply(lambda x: 0 if x in [0, 1] else 1)

# Test Data
df_test_raw = pd.read_csv(content_path+"dev_semeval_parids-labels.csv")
# creating dataset for training classifier
df_cls_test_cols = ['text', 'label', 'par_id', 'keyword', 'country_code']
test_df = df_full.merge(
    df_test_raw.drop(columns=['label']),  # drop label to avoid confusion during merge
    on='par_id', how='inner'
    )
test_df = test_df[df_cls_test_cols]
test_df['text'] = test_df['text'].fillna('')
test_df['label'] = test_df['label'].apply(lambda x: 0 if x in [0, 1] else 1)
test_df.head(2)

## Data Splitting
# Split the data into training and validation sets.

print("--- Splitting the Raw Training Data ---")
internal_train_df, internal_val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=1
)

--- Loading and Preprocessing Data ---
--- Splitting the Raw Training Data ---


In [4]:
# Convert pandas DataFrames to Hugging Face Datasets.

print("--- Converting to Hugging Face Datasets ---")
train_dataset = Dataset.from_pandas(internal_train_df)
val_dataset = Dataset.from_pandas(internal_val_df)

--- Converting to Hugging Face Datasets ---


## Model and Tokenizer

In [5]:
# Load the RoBERTa-base model and tokenizer.

print("--- Loading RoBERTa-base Model and Tokenizer ---")
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

## Model Initialization
def model_init():
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    return model

--- Loading RoBERTa-base Model and Tokenizer ---


Map: 100%|██████████| 1675/1675 [00:00<00:00, 12763.77 examples/s]


## Compute Metrics Function

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary', zero_division=0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

## Training

In [7]:
# Configure and run the training process.

print("--- Training The Baseline Model ---")

learning_rate = 2e-5
weight_decay = 0.01
EPOCHS = 4

training_args = TrainingArguments(
    output_dir="./baseline_weight_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to=[],
    seed=SEED
)


trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print(f"Training for {EPOCHS} epochs ...")
trainer.train()

# Save the trained model and tokenizer.

model_path = "model/"
trainer.save_model(model_path+"baseline_weighting_model")
tokenizer.save_pretrained(model_path+"baseline_weighting_tokenizer")
print(f"Final model saved to {model_path}")

--- Training The Baseline Model ---


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training for 4 epochs ...


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.265100,0.237392,0.300000,0.731707,0.188679
2,0.175400,0.200505,0.531148,0.554795,0.509434
3,0.117600,0.269330,0.552381,0.557692,0.547170
4,0.073700,0.305510,0.540193,0.552632,0.528302


Final model saved to model/


## Prediction on test data

In [12]:
# --- Load Test Data ---

print("--- Loading Test Data ---")
df_test_raw = pd.read_csv(content_path+"dev_semeval_parids-labels.csv")

test_df = df_full.merge(
    df_test_raw.drop(columns=['label']),
    on='par_id', how='inner'
)

test_df = test_df[['text', 'keyword', 'country_code','label', 'par_id']]
test_df['text'] = test_df['text'].fillna('')
test_df['label'] = test_df['label'].apply(lambda x: 0 if x in [0, 1] else 1)
test_dataset = Dataset.from_pandas(test_df)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print("\n--- Generating Predictions on Test Set ---")
# predicting on tokenized_test
test_outputs = trainer.predict(test_dataset=tokenized_test)

# Extract the raw logits
raw_logits = test_outputs.predictions

# Convert logits to final predicted classes (0 or 1) using argmax
# This looks at the two scores for each row and picks the index of the higher one
predicted_classes = np.argmax(raw_logits, axis=1)

# Attach the predictions back tp original pandas DataFrame
test_df['predicted_label'] = predicted_classes

print("\n--- Final Model Performance on Test Set ---")
print(f"Test F1 Score: {test_outputs.metrics['test_f1']:.4f}")
print(f"Test Precision: {test_outputs.metrics['test_precision']:.4f}")
print(f"Test Recall: {test_outputs.metrics['test_recall']:.4f}")

# 5. Check out your new DataFrame!
print("\n--- Sample of Test Data with Predictions ---")
test_df.head()

--- Loading Test Data ---


Map: 100%|██████████| 2094/2094 [00:00<00:00, 5779.79 examples/s]


--- Generating Predictions on Test Set ---



--- Final Model Performance on Test Set ---
Test F1 Score: 0.5985
Test Precision: 0.6094
Test Recall: 0.5879

--- Sample of Test Data with Predictions ---


,text,keyword,country_code,label,par_id,predicted_label
0,"His present "" chambers "" may be quite humble ,...",homeless,ke,1,107,1
1,Krueger recently harnessed that creativity to ...,disabled,us,1,149,0
2,10:41am - Parents of children who died must ge...,poor-families,in,1,151,0
3,When some people feel causing problem for some...,disabled,ng,1,154,1
4,We are alarmed to learn of your recently circu...,poor-families,ca,1,157,0


## Saving the predictions on test_data

In [ ]:
test_df.to_csv("strong_baseline_model_preds.csv", index=False)